# Code

## CUDA Utilities

In [112]:
%%writefile cuda_stuff.cuh
#ifndef cuda_stuff_H
#define cuda_stuff_H

#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>

//MACRO TO DEBUG CUDA FUNCTIONS
/** Error checking,
 *  taken from https://stackoverflow.com/questions/14038589/what-is-the-canonical-way-to-check-for-errors-using-the-cuda-runtime-api
 */
#define gpuErrchk(ans) { gpuAssert((ans), __FILE__, __LINE__); }
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true)
{
   if (code != cudaSuccess)
   {
      fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
      if (abort) exit(code);
   }
}

void device_synchronize();

#endif


Overwriting cuda_stuff.cuh


In [113]:
%%writefile cuda_stuff.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>

#include "cuda_stuff.cuh"

void device_synchronize(){
    gpuErrchk(cudaDeviceSynchronize());
}

Overwriting cuda_stuff.cu


## Matrix Tools

In [114]:
%%writefile fmatrix.cuh
#ifndef fmatrices_H
#define fmatrices_H
#include <stddef.h>

typedef struct {
    float* data;
    size_t cols;
    size_t rows;
} fmatrix;

/* transform matrix index to vector offset
   Since CUDA uses column major,
   nb_rows = number of rows */
#define IDX2C(i,j,nb_rows) (((j)*(nb_rows))+(i))

/* Access element (i,j) of matrix mat */
#define getfm(mat,i,j) (mat.data[IDX2C(i,j,mat.rows)])


size_t fmatrix_elements(fmatrix mat);
size_t fmatrix_size(fmatrix mat);
void fmatrix_init(fmatrix mat, float f);
/** Assert that the matrix is coherent: all fields nonzero. */
void fmatrix_assert();

fmatrix fmatrix_create_on_host(size_t rows, size_t cols);
fmatrix fmatrix_create_on_device(size_t rows, size_t cols);

void fmatrix_data_to_host(fmatrix mat_host, fmatrix mat_device);
void fmatrix_data_to_device(fmatrix mat_host, fmatrix mat_device);

void fmatrix_free_on_host(fmatrix* mat);
void fmatrix_free_on_device(fmatrix* mat);

/** Print the first nb rows of the matrix mat
 *  on the host.
 *  If nb<0, print all rows.
 */
void fmatrix_host_print(fmatrix mat, int nb=-1);

/** Print the first nb rows of the matrix mat
 *  on the device.
 *  If nb<0, print all rows.
 */
void fmatrix_device_print(fmatrix mat, int nb=-1);

#endif


Overwriting fmatrix.cuh


In [115]:
%%writefile fmatrix.cu
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <cuda_runtime.h>

#include "cuda_stuff.cuh"
#include "fmatrix.cuh"

size_t fmatrix_elements(fmatrix mat) {
     return mat.cols*mat.rows;
}

size_t fmatrix_size(fmatrix mat) {
     return fmatrix_elements(mat) * sizeof(float);
}

void fmatrix_init(fmatrix mat, float f) {
    for (int i = 0; i < mat.rows; i++){
        for (int j = 0; j < mat.cols; j++){
            mat.data[IDX2C(i,j,mat.rows)] = f;
    }
  }
}

void fmatrix_assert(fmatrix mat) {
    assert(mat.data);
    assert(mat.cols);
    assert(mat.rows);
}



fmatrix fmatrix_create_on_host(size_t rows, size_t cols) {
    assert(cols>0);
    assert(rows>0);
    fmatrix mat;
    mat.cols = cols;
    mat.rows = rows;
    mat.data = (float*)malloc(fmatrix_size(mat));
    assert(mat.data);
    return mat;
}

fmatrix fmatrix_create_on_device(size_t rows, size_t cols) {
    assert(cols>0);
    assert(rows>0);
    fmatrix mat;
    mat.cols = cols;
    mat.rows = rows;
    gpuErrchk(
        cudaMalloc((void **)&(mat.data), fmatrix_size(mat))
    );
    return mat;
}

void fmatrix_data_to_device(fmatrix mat_host, fmatrix mat_device) {
    fmatrix_assert(mat_host);
    fmatrix_assert(mat_device);
    assert(mat_host.cols==mat_device.cols);
    assert(mat_host.rows==mat_device.rows);
    gpuErrchk(
        cudaMemcpy( mat_device.data, mat_host.data,
                   fmatrix_size(mat_host),
                   cudaMemcpyHostToDevice
                   )
        );
}

void fmatrix_data_to_host(fmatrix mat_host, fmatrix mat_device) {
    fmatrix_assert(mat_host);
    fmatrix_assert(mat_device);
    assert(mat_host.cols==mat_device.cols);
    assert(mat_host.rows==mat_device.rows);
    gpuErrchk(
        cudaMemcpy( mat_host.data, mat_device.data,
                   fmatrix_size(mat_device),
                   cudaMemcpyDeviceToHost
                   )
        );
}

void fmatrix_free_on_host(fmatrix* mat) {
    fmatrix_assert(*mat);
  free(mat->data);
  mat->data = 0;
  mat->cols = 0;
  mat->rows = 0;
}

void fmatrix_free_on_device(fmatrix* mat) {
    fmatrix_assert(*mat);
  gpuErrchk(cudaFree(mat->data));
  mat->data = 0;
  mat->cols = 0;
  mat->rows = 0;
}

void fmatrix_host_print(fmatrix mat, int nb){
    if (nb<0 || nb > mat.rows) {
        nb = mat.rows;
    }
    printf("[\n");
    for (int i = 0 ; i < nb; i++){
      for (int j = 0 ; j<mat.cols; j++){
        printf("%f", getfm(mat,i,j));
        if (j+1<mat.cols) {
          printf(",\t");
        }
      }
      if (i+1<nb) {
        printf(";\n");
      }
    }
    if (nb < mat.rows) {
      printf("\n...\n");
    }
  printf("\n]\n");
}

void fmatrix_device_print(fmatrix mat, int nb){
   // allocate copy
   fmatrix tmp = fmatrix_create_on_host(mat.rows, mat.cols);
   fmatrix_data_to_host(tmp, mat);
   fmatrix_host_print(tmp,nb);
   fmatrix_free_on_host(&tmp);
}



Overwriting fmatrix.cu


## Matrix Math

In [116]:
%%writefile sgemm.cuh
#ifndef sgemm_H
#define sgemm_H

#include <string>
#include "fmatrix.cuh"

void mat_mul(fmatrix A, fmatrix B, fmatrix C, std::string arg);

#endif

Overwriting sgemm.cuh


In [117]:
%%writefile sgemm.cu
#include <stdio.h>
#include <stdlib.h>
#include <string>
#include <time.h>
#include <math.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include "cublas_v2.h"

#include "cuda_stuff.cuh"
#include "sgemm.cuh"
#include "fmatrix.cuh"

#define THREADS_PER_BLOCK 1024
#define TILE_WIDTH 32

using namespace std;

static cublasHandle_t handle;
static int cublas_init = 0;

/* basic matrix multiplication C = alpha*A*B + beta*C on host as reference for the speedup */
void matrixMultiplication_basic_host(float alpha, fmatrix A, fmatrix B, float beta, fmatrix C) {
  float tmp = 0;
  for (int i = 0; i<A.rows; i++){
    for (int j = 0; j<B.cols; j++){
      for (int k = 0; k<A.cols; k++){
        tmp += alpha * getfm(A,i, k) * getfm(B, k, j);
      }
      getfm(C, i, j) = beta * getfm(C, i, j) + tmp;
      tmp = 0;
    }
  }
}

/* TODO : 3 different versions of matrix multiplication C = alpha*A*B + beta*C on device */
__global__
void matmul_basic_kernel(float alpha, float *A, float *B, float beta, float *C, int nb_ColA, int nb_ColB, int nb_LigneA, int nb_LigneB) {
  /* TODO */
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row < nb_LigneA && col < nb_ColB) {
    float sum = 0.0f;
    for (int k = 0; k < nb_ColA; k++) {
      sum += A[k * nb_LigneA + row] * B[col * nb_LigneB + k];
    }
    C[col * nb_LigneA + row] = alpha * sum + beta * C[col * nb_LigneA + row];
  }

}

void matrixMultiplication_basic(float alpha, fmatrix d_A, fmatrix d_B, float beta, fmatrix d_C) {
  // TODO - declaration of dimGrid and dimBlock
  dim3 dimBlock(TILE_WIDTH, TILE_WIDTH, 1);
  dim3 dimGrid((d_B.cols + dimBlock.x - 1) / dimBlock.x,
               (d_A.rows + dimBlock.y - 1) / dimBlock.y,
               1);

  matmul_basic_kernel <<< dimGrid, dimBlock >>> (alpha, d_A.data, d_B.data, beta, d_C.data, d_A.cols, d_B.cols, d_A.rows, d_B.rows);
  gpuErrchk(cudaGetLastError());

}

/**********************/
__global__
void matmul_tiled_kernel(float alpha, float *A, float *B, float beta, float *C, int nb_ColA, int nb_ColB, int nb_LigneA, int nb_LigneB){
  /* TODO */
  __shared__ float As[TILE_WIDTH][TILE_WIDTH];
  __shared__ float Bs[TILE_WIDTH][TILE_WIDTH];

  int row = blockIdx.y * TILE_WIDTH + threadIdx.y;
  int col = blockIdx.x * TILE_WIDTH + threadIdx.x;

  float sum = 0.0f;

  int numTiles = (nb_ColA + TILE_WIDTH - 1) / TILE_WIDTH;

  for (int t = 0; t < numTiles; t++) {
    int tiledColA = t * TILE_WIDTH + threadIdx.x; // column in A
    int tiledRowB = t * TILE_WIDTH + threadIdx.y; // row in B

    // Load A tile
    if (row < nb_LigneA && tiledColA < nb_ColA) {
      As[threadIdx.y][threadIdx.x] = A[tiledColA * nb_LigneA + row];
    } else {
      As[threadIdx.y][threadIdx.x] = 0.0f;
    }

    // Load B tile
    if (tiledRowB < nb_LigneB && col < nb_ColB) {
      Bs[threadIdx.y][threadIdx.x] = B[col * nb_LigneB + tiledRowB];
    } else {
      Bs[threadIdx.y][threadIdx.x] = 0.0f;
    }

    __syncthreads();

    #pragma unroll
    for (int k = 0; k < TILE_WIDTH; k++) {
      sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
    }

    __syncthreads();
  }

  if (row < nb_LigneA && col < nb_ColB) {
    C[col * nb_LigneA + row] = alpha * sum + beta * C[col * nb_LigneA + row];
  }
}



void matrixMultiplication_tiled(float alpha, fmatrix d_A, fmatrix d_B, float beta, fmatrix d_C){
  // TODO - declaration of dimGrid and dimBlock
  dim3 dimBlock(TILE_WIDTH, TILE_WIDTH, 1);
  dim3 dimGrid((d_B.cols + TILE_WIDTH - 1) / TILE_WIDTH,
               (d_A.rows + TILE_WIDTH - 1) / TILE_WIDTH,
               1);

  matmul_tiled_kernel <<< dimGrid, dimBlock >>> (alpha, d_A.data, d_B.data, beta, d_C.data, d_A.cols, d_B.cols, d_A.rows, d_B.rows);
  gpuErrchk(cudaGetLastError());
}

/**********************/
// cublasHandle_t handle;
// int cublas_handle_initialized = 0;

void matrixMultiplication_cublas(float alpha, fmatrix d_A, fmatrix d_B, float beta, fmatrix d_C){
  if (cublas_init == 0){
      cublasStatus_t st = cublasCreate(&handle);
      if (st != CUBLAS_STATUS_SUCCESS) {
        printf("cublasCreate failed with status %d\n", (int)st);
        exit(1);
      }
      cublasSetPointerMode(handle, CUBLAS_POINTER_MODE_HOST);
      cublas_init = 1;
  }
  /* TODO */
  int m = d_A.rows;   // rows de A et C
  int k = d_A.cols;   // cols de A = rows de B
  int n = d_B.cols;   // cols de B et C

  // Column-major standard
  int ldA = m;
  int ldB = k;
  int ldC = m;

  cublasStatus_t stat = cublasSgemm(
      handle,
      CUBLAS_OP_N, CUBLAS_OP_N,
      m, n, k,
      &alpha,
      d_A.data, ldA,
      d_B.data, ldB,
      &beta,
      d_C.data, ldC
  );

  if (stat != CUBLAS_STATUS_SUCCESS) {
    printf("cublasSgemm failed with status %d\n", (int)stat);
    exit(1);
  }
}


/*MAIN SGEMM*/
void gen_mat_mul(float alpha, fmatrix A, fmatrix B, float beta, fmatrix C, std::string arg){
    if (arg == "cpu"){
        matrixMultiplication_basic_host(alpha, A, B, beta, C);
    } else {
      /* kernel function*/
      if (arg == "gpu_basic"){
          matrixMultiplication_basic(alpha, A, B, beta, C);

      } else if (arg == "gpu_tiled"){
          matrixMultiplication_tiled(alpha, A, B, beta, C);

      } else if (arg == "gpu_cublas"){
         matrixMultiplication_cublas(alpha, A, B, beta, C);

      } else{
          printf("Matrix Multiplication argument is Wrong");
          exit(0);
      }
      // wait for everything to finish
      device_synchronize();
    }
}

void mat_mul(fmatrix A, fmatrix B, fmatrix C, std::string arg){
 gen_mat_mul(1.0, A, B, 0.0, C, arg);
}


Overwriting sgemm.cu


# Main

In [118]:
%%writefile main.cu

#include <stdio.h>
#include <stdlib.h>
#include "fmatrix.cuh"
#include "sgemm.cuh"

#define TILE_WIDTH 32
#define SIZE 10

int main(void)
{
  /* Allocate and initialize data on host */
  fmatrix A = fmatrix_create_on_host(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);
  fmatrix_init(A, 1.0);
  fmatrix B = fmatrix_create_on_host(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);
  fmatrix_init(B, 2.0);
  fmatrix C = fmatrix_create_on_host(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);
  fmatrix_init(C, 0.0);

  /* Allocate data on device */
  fmatrix d_A = fmatrix_create_on_device(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);
  fmatrix d_B = fmatrix_create_on_device(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);
  fmatrix d_C = fmatrix_create_on_device(TILE_WIDTH * SIZE, TILE_WIDTH * SIZE);

  /* Transfer A and B on device */
  fmatrix_data_to_device(A, d_A);
  fmatrix_data_to_device(B, d_B);
  fmatrix_data_to_device(C, d_C);

  clock_t start, end;
  float cpu_time_used;
  const int ITERS = 20;

  /* Start calculation "cpu", "gpu_basic", "gpu_tiled", "gpu_cublas" */
  /************** "cpu" *******************/
  start = clock();
  mat_mul(A, B, C, "cpu");
  end = clock();
  cpu_time_used = ((double) (end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("Time taken by CPU in milliseconds: %.2f\n", cpu_time_used);


  /* Result correctness */
  {
    float maxError = 0.0f;
    for (int i = 0; i < TILE_WIDTH * SIZE; i++){
      for (int j = 0; j < TILE_WIDTH * SIZE; j++){
        maxError = max(maxError, abs(getfm(C,i,j)- 2*TILE_WIDTH * SIZE));
      }
    }
    printf("Max error: %f\n", maxError);
  }
  fmatrix_init(C, 0.0);

  /************** "gpu_basic" *******************/
  start = clock();
  mat_mul(d_A, d_B, d_C, "gpu_basic");
  end = clock();
  cpu_time_used = ((double) (end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU basic matrix multiplication in milliseconcs : %.2f\n", cpu_time_used);

  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);

  start = clock();
  for (int it = 0; it < ITERS; it++) {
    mat_mul(d_A, d_B, d_C, "gpu_basic");
  }
  end = clock();

  cpu_time_used = ((double)(end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU basic avg time (ms): %.4f\n", cpu_time_used / ITERS);


  /* Retrieve the result */
  fmatrix_data_to_host(C, d_C);
  /* Result correctness */
  {
    float maxError = 0.0f;
    for (int i = 0; i < TILE_WIDTH * SIZE; i++){
      for (int j = 0; j < TILE_WIDTH * SIZE; j++){
        maxError = max(maxError, abs(getfm(C,i,j)- 2*TILE_WIDTH * SIZE));
      }
    }
    printf("Max error: %f\n", maxError);
  }
  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);


 /************** "gpu_tiled" *******************/
  start = clock();
  mat_mul(d_A, d_B, d_C, "gpu_tiled");
  end = clock();
  cpu_time_used = ((double) (end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU tiled matrix multiplication in milliseconcs : %.2f\n", cpu_time_used);

  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);

  start = clock();
  for (int it = 0; it < ITERS; it++) {
    mat_mul(d_A, d_B, d_C, "gpu_tiled");
  }
  end = clock();

  cpu_time_used = ((double)(end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU tiled avg time (ms): %.4f\n", cpu_time_used / ITERS);


  /* Retrieve the result */
  fmatrix_data_to_host(C, d_C);
  /* Result correctness */
  {
    float maxError = 0.0f;
    for (int i = 0; i < TILE_WIDTH * SIZE; i++){
      for (int j = 0; j < TILE_WIDTH * SIZE; j++){
        maxError = max(maxError, abs(getfm(C,i,j)- 2*TILE_WIDTH * SIZE));
      }
    }
    printf("Max error: %f\n", maxError);
  }
  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);


  /************** "gpu_cublas" *******************/
  for(int warmup = 0; warmup < 5; warmup++){
    mat_mul(d_A, d_B, d_C, "gpu_cublas");
  }
  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);

  start = clock();
  mat_mul(d_A, d_B, d_C, "gpu_cublas");
  end = clock();
  cpu_time_used = ((double) (end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU cuBLAS matrix multiplication in milliseconcs : %.2f\n", cpu_time_used);

  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);

  start = clock();
  for (int it = 0; it < ITERS; it++) {
    mat_mul(d_A, d_B, d_C, "gpu_cublas");
  }
  end = clock();

  cpu_time_used = ((double)(end - start)) * 1000 / CLOCKS_PER_SEC;
  printf("GPU cuBLAS avg time (ms): %.4f\n", cpu_time_used / ITERS);

  /* Retrieve the result */
  fmatrix_data_to_host(C, d_C);
  /* Result correctness */
  {
    float maxError = 0.0f;
    for (int i = 0; i < TILE_WIDTH * SIZE; i++){
      for (int j = 0; j < TILE_WIDTH * SIZE; j++){
        maxError = max(maxError, abs(getfm(C,i,j)- 2*TILE_WIDTH * SIZE));
      }
    }
    printf("Max error: %f\n", maxError);
  }
  fmatrix_init(C, 0.0);
  fmatrix_data_to_device(C, d_C);

  /* Free */
  fmatrix_free_on_host(&A);
  fmatrix_free_on_host(&B);
  fmatrix_free_on_host(&C);
  fmatrix_free_on_device(&d_A);
  fmatrix_free_on_device(&d_B);
  fmatrix_free_on_device(&d_C);
}

Overwriting main.cu


# Compiling

In [119]:
!nvcc -lcublas sgemm.cu  fmatrix.cu  cuda_stuff.cu main.cu -arch=sm_75 -o multiplication

# Experiments

In [120]:
! ./multiplication

Time taken by CPU in milliseconds: 144.98
Max error: 0.000000
GPU basic matrix multiplication in milliseconcs : 1.73
GPU basic avg time (ms): 1.5239
Max error: 0.000000
GPU tiled matrix multiplication in milliseconcs : 0.50
GPU tiled avg time (ms): 0.4285
Max error: 0.000000
GPU cuBLAS matrix multiplication in milliseconcs : 0.10
GPU cuBLAS avg time (ms): 0.0704
Max error: 0.000000


## Remark

We implemented a basic global-memory SGEMM, a tiled shared-memory SGEMM (32x32 blocks), and a cuBLAS SGEMM.

For size 10 (320x320 matrices), the max error is 0 for all methods.

The tiled version is faster than the basic one thanks to data reuse in shared memory.

cuBLAS is the fastest, as expected.

# Debugging
Compile with debugging info on the host (`-g`) and device (`-G`).


In [121]:
!nvcc -g -G -I /usr/local/cuda/samples/common/inc/ -L/usr/local/cuda/include -lcublas -lcusolver sgemm.cu fmatrix.cu cuda_stuff.cu main.cu

Run the debugger cuda-gdb, stopping at the first error that is detected. Shows first the call stack on the GPU, the values of local variables, then the call stack on the host (thread 1).

In [122]:
! printf "set cuda api_failures stop\ncatch throw\nr UNIT\nbt\ninfo locals\nthread 1\nbt\n" > tmp.txt
! cat tmp.txt
! cuda-gdb -batch -x tmp.txt ./a.out

set cuda api_failures stop
catch throw
r UNIT
bt
info locals
thread 1
bt
Catchpoint 1 (throw)
[Thread debugging using libthread_db enabled]
Using host libthread_db library "/lib/x86_64-linux-gnu/libthread_db.so.1".
[New Thread 0x7fffd139a000 (LWP 19183)]
[New Thread 0x7fffd0b99000 (LWP 19184)]
[Detaching after fork from child process 19185]
[New Thread 0x7fffca3de000 (LWP 19190)]
Time taken by CPU in milliseconds: 142.74
Max error: 0.000000
Cuda API error detected: cudaLaunchKernel returned (0xde)
#0  0x00007fffd15ad970 in cudbgReportDriverApiError () from /usr/lib64-nvidia/libcuda.so.1
#1  0x00007fffd182b32b in ?? () from /usr/lib64-nvidia/libcuda.so.1
#2  0x00007fffcb554ba7 in ?? () from /usr/lib64-nvidia/libcudadebugger.so.1
#3  0x00007fffcb531b2e in ?? () from /usr/lib64-nvidia/libcudadebugger.so.1
#4  0x00007fffcb542fda in ?? () from /usr/lib64-nvidia/libcudadebugger.so.1
#5  0x00007fffcb5280d7 in ?? () from /usr/lib64-nvidia/libcudadebugger.so.1
#6  0x00007fffcb69e526 in ?? () fr